In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from transformers import ViTModel

class HybridEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Local Path: ResNet50 for spatial hierarchy
        resnet = models.resnet50(weights='IMAGENET1K_V1')
        self.conv1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu) # 112x112
        self.pool = resnet.maxpool
        self.layer1 = resnet.layer1 # 56x56 (Skip 1)
        self.layer2 = resnet.layer2 # 28x28 (Skip 2)
        self.layer3 = resnet.layer3 # 14x14 (Input to ViT)

        # Global Path: ViT for long-range attention
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.bridge = nn.Conv2d(1024, 768, kernel_size=1) # Channel alignment

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.layer1(self.pool(x1)) # High-res skip
        x3 = self.layer2(x2)            # Mid-res skip
        x4 = self.layer3(x3)            # 14x14 spatial features

        # Reshape for ViT: (B, 1024, 14, 14) -> (B, 196, 768)
        y = self.bridge(x4).flatten(2).transpose(1, 2)
        y = self.vit.encoder(y).last_hidden_state

        # Reshape back: (B, 14, 14, 768)
        y = y.transpose(1, 2).reshape(-1, 768, 14, 14)
        return y, [x1, x2, x3]

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

class HybridUNet(nn.Module):
    def __init__(self, num_classes=4): # Covid, Normal, Bacterial, Viral
        super().__init__()
        self.encoder = HybridEncoder()

        # Decoding stages
        self.up1 = DecoderBlock(768, 512, 512) # From 14x14 to 28x28
        self.up2 = DecoderBlock(512, 256, 256) # From 28x28 to 56x56
        self.up3 = DecoderBlock(256, 64, 64)   # From 56x56 to 112x112

        self.final_up = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.classifier = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        global_feat, skips = self.encoder(x)

        # Iterative fusion of Global (Transformer) and Local (ResNet) data
        d1 = self.up1.upsample(global_feat)
        d1 = torch.cat([d1, skips[2]], dim=1)

        # Final layers generate the segmentation mask
        # ... (implementation continues for up2, up3)
        return self.classifier(self.final_up(d1))

In [ ]:
def calculate_granularity_metrics(preds, masks):
    # preds: (B, 4, H, W), masks: (B, H, W)
    # Calculate Dice score for each class index
    classes = ['Covid', 'Normal', 'Bacterial', 'Viral']
    dice_scores = {}

    for i, name in enumerate(classes):
        p = (preds.argmax(1) == i).float()
        m = (masks == i).float()
        intersection = (p * m).sum()
        dice = (2. * intersection) / (p.sum() + m.sum() + 1e-7)
        dice_scores[name] = dice.item()

    return dice_scores

In [ ]:
import torch.nn.functional as F

class HybridSegmentationLoss(nn.Module):
    def __init__(self, weight=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight)

    def dice_loss(self, inputs, targets, smooth=1e-6):
        inputs = F.softmax(inputs, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=4).permute(0, 3, 1, 2).float()

        intersection = (inputs * targets_one_hot).sum(dim=(2, 3))
        union = inputs.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3))

        dice = (2. * intersection + smooth) / (union + smooth)
        return 1 - dice.mean()

    def forward(self, inputs, targets):
        return self.ce(inputs, targets) + self.dice_loss(inputs, targets)

In [ ]:
from torch.amp import autocast, GradScaler

# Model, Optimizer, and Data Setup
model = HybridUNet(num_classes=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = HybridSegmentationLoss(weight=torch.tensor([1.0, 1.0, 1.3, 1.3]).to(device))
scaler = GradScaler('cuda')

def train_hybrid_segmentation(epochs=20):
    print("🚀 Starting Hybrid U-Net Training for Pathogen Granularity")
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0

        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)

            with autocast('cuda'):
                # Encoder-Decoder Forward Pass
                outputs = model(images)
                loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

        # Validation and Granularity Check
        dice_results = validate_granularity(model, val_loader)
        print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")
        print(f"  Dice - Viral: {dice_results['Viral']:.4f} | Bacterial: {dice_results['Bacterial']:.4f}")

def validate_granularity(model, loader):
    model.eval()
    all_dice = {'Covid': [], 'Normal': [], 'Bacterial': [], 'Viral': []}
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            with autocast('cuda'):
                outputs = model(images)
            # Calculate per-class granularity
            batch_dice = calculate_granularity_metrics(outputs, masks)
            for k, v in batch_dice.items():
                all_dice[k].append(v)
    return {k: sum(v)/len(v) for k, v in all_dice.items()}

In [ ]:
import matplotlib.pyplot as plt

def plot_segmentation_results(image, mask, prediction):
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(image.permute(1, 2, 0).cpu())
    ax[0].set_title("Original Chest X-Ray")

    ax[1].imshow(mask.cpu(), cmap='jet')
    ax[1].set_title("Ground Truth (Pathogen Zones)")

    ax[2].imshow(prediction.argmax(0).cpu(), cmap='jet')
    ax[2].set_title("Hybrid U-Net Prediction")
    plt.show()